# DWBI Week 2 — DW Lifecycle & Architecture
**Ali Raza** · `su92-bsdsm-f23-018` · **BSDS-7A**

Case: **PakFreight** — three transport modes, three files, three totals, zero trust.
The week ends with one trusted table: `Shipments`, 34 rows, 578,650 kg.

## Task 1 — The three tiers, and what a warehouse must be

### 1. The three tiers

| Tier | One-line job | In the PakFreight case | Must NOT happen here |
|---|---|---|---|
| **Tier 1 — Staging** | Quality checkpoint: land raw files, fix them, record every fix | `dd/mm/yyyy` → real dates (UK locale); tons × 1000; `RL-` prefix; drop duplicate `rd-505`, blank row, TOTAL row — each named | Raw files edited in place; fixes done by hand with no recorded step; unvalidated data passed downstream |
| **Tier 2 — Storage** | One integrated, historical store (EDW / ODS) | A single `Shipments` table: 34 rows × 8 columns, 578,650 kg, every row traceable to file + row + transformation | Three departmental totals living side by side; business logic or report formatting baked into stored data |
| **Tier 3 — Presentation** | Star-schema marts shaped for questions | Card `SUM(WeightKG)` = 578,650; bar by `SourceSystem`; status counts 22 / 8 / 4 | Raw `_stg` queries visible to users; unvalidated figures published; a "report" that is really a spreadsheet dump |

![Three tiers](charts/t1_tiers.png)

### 2. Inmon's four properties, applied to PakFreight

| Property | Satisfied — example from this case | Broken — example from this case |
|---|---|---|
| **Subject-oriented** | The `Shipments` table is organised around the *shipment* subject, not around marine/rail/road departments | Each department's mart is organised around its own file — Operations thinks in "marine rows", Finance in "rail rows" |
| **Integrated** | One contract: `ShipmentID, ShipDate, Origin, Destination, WeightKG, CostPKR, Status, SourceSystem` — every source bent into it | Tons vs kilograms, `dd/mm/yyyy` vs real dates, `DELIVERED` vs `delivered` vs `DELAYED` — the same reality encoded three ways |
| **Time-variant** | Keeping every February shipment with its `ShipDate` so the CFO can compare cost per tonne-km across three years | Overwriting a customer's old address with the new one — history is destroyed, and no past report can ever be reproduced |
| **Non-volatile** | Once loaded, a shipment row is never updated in place; corrections arrive as new rows / new loads | Editing `marine_shipments.csv` in Excel and saving over it — the evidence is destroyed and Refresh can never reproduce the answer |

### 3. EDW or ODS?

| | Requirement | Verdict | One line |
|---|---|---|---|
| a | Dispatcher needs truck positions every 2 minutes | **ODS** | Operational, current detail — the EDW's historical batches are far too slow |
| b | CFO compares freight cost per tonne-km across three years | **EDW** | Integrated history is exactly what the warehouse keeps |
| c | Agent checks whether consignment 45017 left today | **ODS** | Today's operational state, not history |
| d | Board reviews mode profitability since 2023 | **EDW** | Multi-year, subject-oriented analysis |

### 4. Independent vs dependent marts

![Independent vs dependent](charts/t1_marts.png)

Three sentences on why the first arrangement produces three answers to one question: each department builds straight from its own file, so each inherits its file's private assumptions — Operations keeps the TOTAL row, Finance reads tons as kilograms, Sales keeps the duplicate. With no shared staging and no shared rules, "total weight" means something different in each mart. Three honest teams, three honest sums, three different numbers — that is the stovepipe problem, and only a dependent design (one staging area, one EDW, marts derived from it) removes it.

### 5. One sentence on staging

Raw files pass through staging rather than loading straight into the model because staging is where evidence is preserved and every fix is recorded — without it, a wrong number can never be traced back to the file, row and transformation that produced it.

### The question to answer

Overwriting a customer's old address with the new one breaks **time-variant**: the warehouse loses the ability to say where the customer *was* at any point in time. What it should have done instead is keep history — a slowly changing dimension (Type 2: a new row with effective dates), so both the old and the new address remain queryable.

In [ ]:
# Sanity check used throughout this notebook series: the one trusted number
MARINE_KG, RAIL_KG, ROAD_KG = 206500, 295250, 76900
truth = MARINE_KG + RAIL_KG + ROAD_KG
assert truth == 578650, truth
print(f"One staging area -> {truth:,} kg = {truth/1000:.2f} t  (the number every tier must agree on)")